In [1]:
import sys
sys.path.insert(0, '../..')

import ast
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp

Path('../../data/features').mkdir(parents=True, exist_ok=True)

PROC = '../../data/processed/'
FEAT = '../../data/features/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
movies  = pd.read_csv(PROC + 'movies_master.csv')
ratings = pd.read_csv(PROC + 'ratings_cleaned.csv')

# Parse list columns back from string representation
# They were saved as strings in CSV
import ast

def safe_parse_list(val):
    try:
        result = ast.literal_eval(str(val))
        return result if isinstance(result, list) else []
    except:
        return []

movies['genres_list']  = movies['genres_list'].apply(safe_parse_list)
movies['cast_names']   = movies['cast_names'].apply(safe_parse_list)
movies['keyword_list'] = movies['keyword_list'].apply(safe_parse_list)

# Fill text nulls
movies['overview']  = movies['overview'].fillna('')
movies['tagline']   = movies['tagline'].fillna('')
movies['director']  = movies['director'].fillna('')
movies['title']     = movies['title'].fillna('')

print(f"Movies  : {movies.shape}")
print(f"Ratings : {ratings.shape}")
print(f"\nSample movie:")
print(movies[['title', 'genres_list',
              'cast_names', 'director']].iloc[0])

Movies  : (45454, 24)
Ratings : (100004, 4)

Sample movie:
title                                    Toy Story
genres_list            [Animation, Comedy, Family]
cast_names     [Tom Hanks, Tim Allen, Don Rickles]
director                             John Lasseter
Name: 0, dtype: object


In [4]:
#Build Combined Text SoupWhy: A single rich text field per movie that combines all content signals. 
#This is what gets fed into sentence-transformers (e5-large) and CLIP text encoder in Week 2. Better than using each field separately.

def build_text_soup(row):
    """
    Combine all text signals into one rich string per movie.
    Weight important fields by repeating them.
    """
    parts = []

    # Title repeated 3x — most important signal
    if row['title']:
        parts.extend([str(row['title'])] * 3)

    # Overview — narrative content
    if row['overview']:
        parts.append(str(row['overview']))

    # Tagline
    if row['tagline']:
        parts.append(str(row['tagline']))

    # Genres repeated 2x — strong content signal
    if row['genres_list']:
        genre_str = ' '.join(row['genres_list'])
        parts.extend([genre_str] * 2)

    # Director repeated 2x — auteur signal
    if row['director']:
        parts.extend([str(row['director'])] * 2)

    # Top cast
    if row['cast_names']:
        parts.append(' '.join(row['cast_names']))

    # Keywords
    if row['keyword_list']:
        parts.append(' '.join(row['keyword_list'][:10]))

    return ' '.join(parts).lower().strip()

movies['text_soup'] = movies.apply(build_text_soup, axis=1)

print("Text soup built ✅")
print(f"\nExample text soup for '{movies['title'].iloc[0]}':")
print(movies['text_soup'].iloc[0][:300])
print(f"\nAvg text soup length: "
      f"{movies['text_soup'].str.len().mean():.0f} chars")

Text soup built ✅

Example text soup for 'Toy Story':
toy story toy story toy story led by woody, andy's toys live happily in his room until andy's birthday brings buzz lightyear onto the scene. afraid of losing his place in andy's heart, woody plots against buzz. but when circumstances separate buzz and woody from their owner, the duo eventually learn

Avg text soup length: 522 chars


In [6]:
# TF-IDF Features
#Why: TF-IDF is your baseline content similarity method. 
#Used in Week 2 as a comparison baseline against the modern e5-large embeddings. Also used directly in the hybrid scorer.

print("Building TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features=10_000,    # top 10K terms
    ngram_range=(1, 2),     # unigrams + bigrams
    min_df=2,               # ignore terms in < 2 movies
    max_df=0.95,            # ignore terms in > 95% movies
    sublinear_tf=True,      # apply log normalization
    strip_accents='unicode',
    analyzer='word',
)

tfidf_matrix = tfidf.fit_transform(movies['text_soup'])

print(f"TF-IDF matrix shape : {tfidf_matrix.shape}")
print(f"Vocabulary size     : {len(tfidf.vocabulary_):,}")
print(f"Matrix density      : "
      f"{tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]) * 100:.3f}%")

# Save sparse matrix + vocabulary
sp.save_npz(FEAT + 'tfidf_matrix.npz', tfidf_matrix)

import joblib
joblib.dump(tfidf, FEAT + 'tfidf_vectorizer.joblib')

print("✅ TF-IDF matrix saved")


Building TF-IDF features...
TF-IDF matrix shape : (45454, 10000)
Vocabulary size     : 10,000
Matrix density      : 0.643%
✅ TF-IDF matrix saved


In [7]:
# Genre Multi-Hot Encoding
#Why: Multi-hot genre vectors let you compute genre-based similarity directly. Used in the hybrid ranker and diversity re-ranking in Week 4.

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres_list'])
genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=movies.index
)

print(f"Genre matrix shape : {genre_df.shape}")
print(f"Genres             : {list(mlb.classes_)}")
print(f"\nGenre distribution (% movies with genre):")
genre_pct = (genre_df.sum() / len(genre_df) * 100).sort_values(
    ascending=False)
for genre, pct in genre_pct.head(10).items():
    print(f"  {genre:<20} {pct:.1f}%")

# Add movie_id for joining
genre_df['movie_id'] = movies['id'].values

# Save
genre_df.to_csv(FEAT + 'genre_features.csv', index=False)
joblib.dump(mlb, FEAT + 'genre_encoder.joblib')

print("\n✅ Genre features saved")

Genre matrix shape : (45454, 20)
Genres             : ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']

Genre distribution (% movies with genre):
  Drama                44.6%
  Comedy               29.0%
  Thriller             16.8%
  Romance              14.8%
  Action               14.5%
  Horror               10.3%
  Crime                9.5%
  Documentary          8.7%
  Adventure            7.7%
  Science Fiction      6.7%

✅ Genre features saved


In [8]:
## Numerical Features
## Why: Budget, revenue, runtime, vote stats are strong signals for popularity and quality. 
#Normalised to [0,1] so they do not dominate gradient-based models.

num_features = movies[[
    'id', 'budget', 'revenue', 'runtime',
    'vote_average', 'vote_count', 'popularity'
]].copy()

# Log transform skewed columns before normalising
for col in ['budget', 'revenue', 'vote_count', 'popularity']:
    num_features[col] = np.log1p(
        num_features[col].fillna(0))

# Fill remaining nulls with median
for col in ['runtime', 'vote_average']:
    num_features[col] = num_features[col].fillna(
        num_features[col].median())

# Normalise to [0, 1]
cols_to_scale = [
    'budget', 'revenue', 'runtime',
    'vote_average', 'vote_count', 'popularity'
]
scaler = MinMaxScaler()
num_features[cols_to_scale] = scaler.fit_transform(
    num_features[cols_to_scale])

print(f"Numerical features shape: {num_features.shape}")
print(num_features.describe().round(3))

num_features.to_csv(FEAT + 'numerical_features.csv', index=False)
joblib.dump(scaler, FEAT + 'numerical_scaler.joblib')

print("\n✅ Numerical features saved")

Numerical features shape: (45454, 7)
               id     budget    revenue    runtime  vote_average  vote_count  \
count   45454.000  45454.000  45454.000  45454.000     45454.000   45454.000   
mean   108348.389      0.152      0.120      0.075         0.562       0.278   
std    112445.976      0.315      0.277      0.030         0.192       0.181   
min         2.000      0.000      0.000      0.000         0.000       0.000   
25%     26447.500      0.000      0.000      0.068         0.500       0.145   
50%     60004.000      0.000      0.000      0.076         0.600       0.251   
75%    157304.000      0.000      0.000      0.085         0.680       0.372   
max    469172.000      1.000      1.000      1.000         1.000       1.000   

       popularity  
count   45454.000  
mean        0.155  
std         0.127  
min         0.000  
25%         0.052  
50%         0.120  
75%         0.245  
max         1.000  

✅ Numerical features saved


In [9]:
## Temporal Features
#Why: Users have recency preferences. 
#A 2015 movie and a 1950 movie are fundamentally different recommendation contexts. Temporal features capture this.

movies['release_date'] = pd.to_datetime(
    movies['release_date'], errors='coerce')

reference_date = pd.Timestamp('2017-07-01')  # dataset cutoff

temporal = movies[['id', 'year']].copy()

# Decade encoding
temporal['decade'] = (movies['year'] // 10 * 10).astype('Int64')

# Days since release (relative age)
temporal['days_since_release'] = (
    reference_date - movies['release_date']
).dt.days.fillna(-1).astype(int)

# Era encoding
def get_era(year):
    if pd.isna(year):  return 'unknown'
    year = int(year)
    if year < 1950:    return 'classic'
    elif year < 1970:  return 'golden_age'
    elif year < 1990:  return 'modern_classic'
    elif year < 2000:  return 'late_modern'
    elif year < 2010:  return 'contemporary'
    else:              return 'recent'

temporal['era'] = movies['year'].apply(get_era)

# One-hot encode era
era_dummies = pd.get_dummies(
    temporal['era'], prefix='era')
temporal = pd.concat([temporal, era_dummies], axis=1)

print(f"Temporal features shape: {temporal.shape}")
print(temporal.head())
print(f"\nEra distribution:")
print(temporal['era'].value_counts())

temporal.to_csv(FEAT + 'temporal_features.csv', index=False)
print("\n✅ Temporal features saved")

Temporal features shape: (45454, 12)
      id    year  decade  days_since_release          era  era_classic  \
0    862  1995.0    1990                7915  late_modern        False   
1   8844  1995.0    1990                7869  late_modern        False   
2  15602  1995.0    1990                7862  late_modern        False   
3  31357  1995.0    1990                7862  late_modern        False   
4  11862  1995.0    1990                8177  late_modern        False   

   era_contemporary  era_golden_age  era_late_modern  era_modern_classic  \
0             False           False             True               False   
1             False           False             True               False   
2             False           False             True               False   
3             False           False             True               False   
4             False           False             True               False   

   era_recent  era_unknown  
0       False        False  
1  

In [14]:
## Movie Interaction Features (From Ratings)
movie_stats = ratings.groupby('movieId').agg(
    rating_count  = ('rating', 'count'),
    rating_mean   = ('rating', 'mean'),
    rating_std    = ('rating', 'std'),
    rating_median = ('rating', 'median'),
    unique_users  = ('userId', 'nunique'),
).reset_index()

movie_stats['rating_std']       = movie_stats['rating_std'].fillna(0)
movie_stats['log_rating_count'] = np.log1p(movie_stats['rating_count'])

# Use rank-based percentile instead of qcut
# avoids bin edge uniqueness problem entirely
movie_stats['popularity_pct'] = movie_stats['rating_count'].rank(pct=True)

def popularity_tier(pct):
    if pct <= 0.25:   return 'cold'
    elif pct <= 0.50: return 'low'
    elif pct <= 0.75: return 'medium'
    else:             return 'hot'

movie_stats['popularity_tier'] = movie_stats[
    'popularity_pct'].apply(popularity_tier)

print(f"Movie interaction features: {movie_stats.shape}")
print(movie_stats.describe().round(3))
print(f"\nPopularity tier distribution:")
print(movie_stats['popularity_tier'].value_counts())

movie_stats.to_csv(FEAT + 'movie_interaction_features.csv',
                   index=False)
print("\n✅ Movie interaction features saved")

Movie interaction features: (9066, 9)
          movieId  rating_count  rating_mean  rating_std  rating_median  \
count    9066.000      9066.000     9066.000    9066.000       9066.000   
mean    30772.100        11.031        3.292       0.579          3.331   
std     40418.421        24.051        0.882       0.526          0.918   
min         1.000         1.000        0.500       0.000          0.500   
25%      2829.750         1.000        2.844       0.000          3.000   
50%      6248.000         3.000        3.500       0.672          3.500   
75%     55827.500         9.000        3.966       0.966          4.000   
max    163949.000       341.000        5.000       3.182          5.000   

       unique_users  log_rating_count  popularity_pct  
count      9066.000          9066.000        9066.000  
mean         11.031             1.689           0.500  
std          24.051             1.091           0.283  
min           1.000             0.693           0.169  
25%   

In [15]:
# User Features

user_stats = ratings.groupby('userId').agg(
    rating_count  = ('rating', 'count'),
    rating_mean   = ('rating', 'mean'),
    rating_std    = ('rating', 'std'),
    unique_movies = ('movieId', 'nunique'),
).reset_index()

user_stats['rating_std'] = user_stats['rating_std'].fillna(0)

# Activity tier — rank based, avoids bin edge uniqueness problem
user_stats['activity_pct'] = user_stats['rating_count'].rank(pct=True)

def activity_tier(pct):
    if pct <= 0.25:   return 'inactive'
    elif pct <= 0.50: return 'casual'
    elif pct <= 0.75: return 'active'
    else:             return 'power'

user_stats['activity_tier'] = user_stats[
    'activity_pct'].apply(activity_tier)

# Rating tendency
def rating_tendency(mean):
    if mean <= 2.5:   return 'critical'
    elif mean <= 3.5: return 'neutral'
    else:             return 'positive'

user_stats['rating_tendency'] = user_stats[
    'rating_mean'].apply(rating_tendency)

# Top genre per user
ratings_with_genres = ratings.merge(
    movies[['movieId', 'genres_list']],
    on='movieId', how='left'
)
ratings_with_genres['genres_list'] = \
    ratings_with_genres['genres_list'].apply(safe_parse_list)

genre_exploded = ratings_with_genres.explode('genres_list')
genre_exploded = genre_exploded[
    genre_exploded['genres_list'].notna() &
    (genre_exploded['genres_list'] != '')
]

top_genre = genre_exploded.groupby(
    ['userId', 'genres_list']
)['rating'].count().reset_index()

top_genre = top_genre.loc[
    top_genre.groupby('userId')['rating'].idxmax()
][['userId', 'genres_list']].rename(
    columns={'genres_list': 'top_genre'})

user_stats = user_stats.merge(
    top_genre, on='userId', how='left')

print(f"User features shape: {user_stats.shape}")
print(user_stats.describe().round(3))
print(f"\nActivity tier distribution:")
print(user_stats['activity_tier'].value_counts())
print(f"\nRating tendency distribution:")
print(user_stats['rating_tendency'].value_counts())

user_stats.to_csv(FEAT + 'user_features.csv', index=False)
print("\n✅ User features saved")

User features shape: (671, 9)
        userId  rating_count  rating_mean  rating_std  unique_movies  \
count  671.000       671.000      671.000     671.000        671.000   
mean   336.000       149.037        3.658       0.940        149.037   
std    193.845       231.227        0.471       0.240        231.227   
min      1.000        20.000        1.333       0.192         20.000   
25%    168.500        37.000        3.396       0.772         37.000   
50%    336.000        71.000        3.675       0.922         71.000   
75%    503.500       161.000        3.984       1.089        161.000   
max    671.000      2391.000        4.949       1.892       2391.000   

       activity_pct  
count       671.000  
mean          0.501  
std           0.289  
min           0.022  
25%           0.251  
50%           0.500  
75%           0.751  
max           1.000  

Activity tier distribution:
activity_tier
casual      170
power       169
inactive    166
active      166
Name: count, dty

In [16]:
# Build Interaction Matrix

from scipy.sparse import csr_matrix

# Create user and movie index mappings
user_ids  = sorted(ratings['userId'].unique())
movie_ids = sorted(ratings['movieId'].unique())

user2idx  = {u: i for i, u in enumerate(user_ids)}
movie2idx = {m: i for i, m in enumerate(movie_ids)}

# Map to indices
row = ratings['userId'].map(user2idx).values
col = ratings['movieId'].map(movie2idx).values
val = ratings['rating'].values

# Build sparse matrix
interaction_matrix = csr_matrix(
    (val, (row, col)),
    shape=(len(user_ids), len(movie_ids))
)

sparsity = 1 - (interaction_matrix.nnz /
                (interaction_matrix.shape[0] *
                 interaction_matrix.shape[1]))

print(f"Interaction matrix shape : {interaction_matrix.shape}")
print(f"Non-zero entries         : {interaction_matrix.nnz:,}")
print(f"Sparsity                 : {sparsity*100:.4f}%")

# Save matrix and mappings
sp.save_npz(FEAT + 'interaction_matrix.npz',
            interaction_matrix)

mappings = {
    'user2idx':  user2idx,
    'movie2idx': movie2idx,
    'idx2user':  {i: u for u, i in user2idx.items()},
    'idx2movie': {i: m for m, i in movie2idx.items()},
}
joblib.dump(mappings, FEAT + 'id_mappings.joblib')

print("\n✅ Interaction matrix and mappings saved")

Interaction matrix shape : (671, 9066)
Non-zero entries         : 100,004
Sparsity                 : 98.3561%

✅ Interaction matrix and mappings saved


In [17]:
## Build Master Feature Table

# Start with movies master
feature_table = movies[[
    'id', 'movieId', 'title', 'text_soup',
    'genres_list', 'cast_names', 'keyword_list',
    'director', 'original_language', 'year'
]].copy()

# Join numerical features
feature_table = feature_table.merge(
    num_features.drop(columns=[]),
    on='id', how='left'
)

# Join temporal features
feature_table = feature_table.merge(
    temporal[['id', 'decade', 'era', 'days_since_release']],
    on='id', how='left'
)

# Join movie interaction stats
feature_table = feature_table.merge(
    movie_stats[['movieId', 'rating_count', 'rating_mean',
                 'rating_std', 'log_rating_count',
                 'popularity_tier']],
    on='movieId', how='left'
)

print(f"Master feature table shape: {feature_table.shape}")
print(f"Columns: {list(feature_table.columns)}")

# Only keep movies that have a movieId (appear in ratings)
feature_table_rated = feature_table[
    feature_table['movieId'].notna()
].copy()

print(f"\nMovies with ratings: {len(feature_table_rated):,}")
print(f"Movies without ratings (metadata only): "
      f"{feature_table['movieId'].isna().sum():,}")

# Save both
feature_table.to_csv(
    FEAT + 'movie_features_all.csv', index=False)
feature_table_rated.to_csv(
    FEAT + 'movie_features_rated.csv', index=False)

print("\n✅ Master feature tables saved")

Master feature table shape: (45646, 24)
Columns: ['id', 'movieId', 'title', 'text_soup', 'genres_list', 'cast_names', 'keyword_list', 'director', 'original_language', 'year', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'decade', 'era', 'days_since_release', 'rating_count', 'rating_mean', 'rating_std', 'log_rating_count', 'popularity_tier']

Movies with ratings: 45,646
Movies without ratings (metadata only): 0

✅ Master feature tables saved


In [18]:
## Register Features In Feast

##Why: Feast is your feature store. Registering features here means every model — training AND serving — gets the exact same feature values. 
##This eliminates training-serving skew.

# Create Feast feature store configuration
feast_config = {
    "project":  "production_recsys",
    "registry": "feast/feature_store/registry.db",
    "provider": "local",
    "online_store": {
        "type": "redis",
        "connection_string": "localhost:6379"
    },
    "offline_store": {
        "type": "file"
    }
}

import yaml
feast_dir = Path('../../feast/feature_store')
feast_dir.mkdir(parents=True, exist_ok=True)

with open(feast_dir / 'feature_store.yaml', 'w') as f:
    yaml.dump(feast_config, f)

# Save feature metadata for Feast registration
feature_registry = {
    "movie_features": {
        "entity":   "movie_id",
        "features": [
            "title", "text_soup", "genres_list",
            "vote_average", "vote_count", "popularity",
            "budget", "revenue", "runtime",
            "year", "era", "rating_count",
            "rating_mean", "log_rating_count"
        ],
        "source": str(FEAT + 'movie_features_rated.csv')
    },
    "user_features": {
        "entity":   "user_id",
        "features": [
            "rating_count", "rating_mean",
            "unique_movies", "activity_tier",
            "rating_tendency", "top_genre"
        ],
        "source": str(FEAT + 'user_features.csv')
    }
}

with open(feast_dir / 'feature_registry.json', 'w') as f:
    json.dump(feature_registry, f, indent=2)

print("✅ Feast feature store config saved")
print(f"   Location: {feast_dir}")


✅ Feast feature store config saved
   Location: ../../feast/feature_store


In [19]:
# Final Summary

import os

print("FEATURE ENGINEERING COMPLETE")
print("=" * 55)
print(f"\nFiles saved to data/features/:")

for f in sorted(os.listdir(FEAT)):
    size = os.path.getsize(FEAT + f) / 1024 / 1024
    print(f"  {f:<45} {size:.1f} MB")

print(f"""
WHAT EACH FILE IS USED FOR
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tfidf_matrix.npz          → Week 2 content baseline
tfidf_vectorizer.joblib   → Week 2 transform new movies
genre_features.csv        → Week 2 genre similarity
genre_encoder.joblib      → Week 2 encode new movies
numerical_features.csv    → Week 3 HSTU ranker input
numerical_scaler.joblib   → Week 5 serving normalisation
temporal_features.csv     → Week 3 HSTU ranker input
movie_interaction_features→ Week 2 popularity signal
user_features.csv         → Feast → Week 3 ranker
interaction_matrix.npz    → Week 2 CF models direct input
id_mappings.joblib        → all weeks userId↔index
movie_features_all.csv    → all models — master table
movie_features_rated.csv  → models that need ratings
""")

FEATURE ENGINEERING COMPLETE

Files saved to data/features/:
  genre_encoder.joblib                          0.0 MB
  genre_features.csv                            2.0 MB
  id_mappings.joblib                            0.3 MB
  interaction_matrix.npz                        0.2 MB
  movie_features_all.csv                        35.5 MB
  movie_features_rated.csv                      35.5 MB
  movie_interaction_features.csv                0.7 MB
  numerical_features.csv                        3.8 MB
  numerical_scaler.joblib                       0.0 MB
  temporal_features.csv                         3.3 MB
  tfidf_matrix.npz                              26.0 MB
  tfidf_vectorizer.joblib                       21.6 MB
  user_features.csv                             0.1 MB

WHAT EACH FILE IS USED FOR
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tfidf_matrix.npz          → Week 2 content baseline
tfidf_vectorizer.joblib   → Week 2 transform new movies
genre_features.csv        → Week 2